# TRÍ TUỆ NHÂN TẠO – BÀI GIẢNG THỰC HÀNH
## Buổi 4: Biểu diễn tri thức và suy diễn (10 bài tập Python trên Google Colab)
* **Họ và tên:** Nguyễn Trung Tín
* **MSSV:** 25110200091
* **Khoa:** Công nghệ Thông tin & Trí tuệ Nhân tạo (CNTT-TTNT)
* **Khóa:** 25
* **Môn học:** Trí tuệ nhân tạo (Thực hành Lập trình Python)
* **Môi trường thực hiện:** Google Colab (Python 3)


### Cài đặt các thư viện cần thiết
Ô code đầu tiên theo yêu cầu của bài giảng thực hành Buổi 4.


In [ ]:
!pip -q install sympy networkx matplotlib pandas


---
## Bài 1: Bảng chân trị và phân loại công thức (Mức độ: Cơ bản)
**Mục tiêu:** Biểu diễn công thức mệnh đề bằng hàm Python, sinh bảng chân trị và phân loại công thức: hằng đúng, mâu thuẫn hay thỏa được.

**Các công thức kiểm tra:**
* (a) $((P \rightarrow Q) \wedge (Q \rightarrow R)) \rightarrow (P \rightarrow R)$
* (b) $(P \vee Q) \wedge \neg P \wedge \neg Q$
* (c) $(P \rightarrow Q) \wedge (Q \rightarrow \neg P)$
* (d) $P \leftrightarrow (Q \wedge \neg P)$


In [ ]:
import itertools
import pandas as pd

imp = lambda a, b: (not a) or b # a -> b
iff = lambda a, b: a == b       # a <-> b

def bang_chan_tri(bien, f):
    """bien: list ten bien, f: ham nhan dict {ten: bool} -> bool.
    Tra ve DataFrame gom cac cot bien va cot 'KQ'."""
    hang = []
    # duyet itertools.product([False, True], repeat=len(bien))
    # moi to hop -> dict env -> them dong (cac gia tri + f(env))
    for vals in itertools.product([False, True], repeat=len(bien)):
        env = dict(zip(bien, vals))
        hang.append(list(vals) + [f(env)])
    return pd.DataFrame(hang, columns=bien + ['KQ'])

def phan_loai(bien, f):
    """Tra ve 'hang dung' | 'mau thuan' | 'thoa duoc'."""
    kqs = [f(dict(zip(bien, vals))) for vals in itertools.product([False, True], repeat=len(bien))]
    if all(kqs):
        return 'hang dung'
    elif not any(kqs):
        return 'mau thuan'
    else:
        return 'thoa duoc'

# (a)
f1 = lambda e: imp(imp(e['P'], e['Q']) and imp(e['Q'], e['R']), imp(e['P'], e['R']))
print("--- Bang chan tri cau (a) ---")
print(bang_chan_tri(['P', 'Q', 'R'], f1))
pl_a = phan_loai(['P', 'Q', 'R'], f1)
print("Phan loai (a):", pl_a)
assert pl_a == 'hang dung'

# (b) (P v Q) ^ ~P ^ ~Q
f2 = lambda e: (e['P'] or e['Q']) and (not e['P']) and (not e['Q'])
print("\n--- Bang chan tri cau (b) ---")
print(bang_chan_tri(['P', 'Q'], f2))
pl_b = phan_loai(['P', 'Q'], f2)
print("Phan loai (b):", pl_b)
assert pl_b == 'mau thuan'

# (c) (P -> Q) ^ (Q -> ~P)
f3 = lambda e: imp(e['P'], e['Q']) and imp(e['Q'], not e['P'])
print("\n--- Bang chan tri cau (c) ---")
print(bang_chan_tri(['P', 'Q'], f3))
pl_c = phan_loai(['P', 'Q'], f3)
print("Phan loai (c):", pl_c)
assert pl_c == 'thoa duoc'

# (d) P <-> (Q ^ ~P)
f4 = lambda e: iff(e['P'], e['Q'] and (not e['P']))
print("\n--- Bang chan tri cau (d) ---")
print(bang_chan_tri(['P', 'Q'], f4))
pl_d = phan_loai(['P', 'Q'], f4)
print("Phan loai (d):", pl_d)
assert pl_d == 'thoa duoc'


**Giải thích kết quả câu (c) và (d):**
* **Câu (c) $(P \rightarrow Q) \wedge (Q \rightarrow \neg P)$:** Là công thức **thỏa được** vì có ít nhất một mô hình làm công thức đúng (cụ thể: khi $P = False, Q = False$ thì cả hai phép kéo theo đều nhận giá trị True, dẫn đến kết quả hội là True; tuy nhiên khi $P = True, Q = True$ thì $Q \rightarrow \neg P$ là False nên công thức không phải hằng đúng).
* **Câu (d) $P \leftrightarrow (Q \wedge \neg P)$:** Là công thức **thỏa được** vì khi $P = False, Q = False$, vế trái $P = False$ và vế phải $Q \wedge \neg P = False$, hai vế tương đương nhau cho ra kết quả True (nó không phải hằng đúng vì khi $P = True$, vế trái là True nhưng vế phải luôn False do chứa $\neg P$, khiến phép tương đương False).


---
## Bài 2: Tương đương logic và hệ quả logic (Mức độ: Cơ bản)
**Mục tiêu:** Cài đặt kiểm tra mô hình (model checking) để chứng minh $f \equiv g$ và $KB \models \alpha$.


In [ ]:
import itertools

def cac_mo_hinh(bien):
    for vals in itertools.product([False, True], repeat=len(bien)):
        yield dict(zip(bien, vals))

def tuong_duong(bien, f, g):
    """Tra ve True neu f(m) == g(m) voi moi mo hinh m."""
    return all(f(m) == g(m) for m in cac_mo_hinh(bien))

def he_qua(bien, KB, alpha):
    """Tra ve True neu moi mo hinh thoa KB(m) deu thoa alpha(m)."""
    return all(alpha(m) for m in cac_mo_hinh(bien) if KB(m))

imp = lambda a, b: (not a) or b

# (a) Kiem tra De Morgan: ~(P ^ Q) == ~P v ~Q
dm_left = lambda e: not (e['P'] and e['Q'])
dm_right = lambda e: (not e['P']) or (not e['Q'])
print("Luat De Morgan ~(P ^ Q) == ~P v ~Q:", tuong_duong(['P', 'Q'], dm_left, dm_right))
assert tuong_duong(['P', 'Q'], dm_left, dm_right) == True

# Luat phan dao: P -> Q == ~Q -> ~P
pd_left = lambda e: imp(e['P'], e['Q'])
pd_right = lambda e: imp(not e['Q'], not e['P'])
print("Luat phan dao P -> Q == ~Q -> ~P:", tuong_duong(['P', 'Q'], pd_left, pd_right))
assert tuong_duong(['P', 'Q'], pd_left, pd_right) == True

# Thu mot cap khong tuong duong: P -> Q va Q -> P
khong_td = tuong_duong(['P', 'Q'], lambda e: imp(e['P'], e['Q']), lambda e: imp(e['Q'], e['P']))
print("P -> Q co tuong duong Q -> P khong?:", khong_td)
assert khong_td == False

# (b) Kiem tra he qua KB |= alpha
KB = lambda e: imp(e['P'], e['Q']) and imp(e['Q'], e['R']) and e['P']
alpha = lambda e: e['R']
print("KB |= R ?", he_qua(['P', 'Q', 'R'], KB, alpha)) # True
assert he_qua(['P', 'Q', 'R'], KB, alpha) == True

# (c) Hiep si va ke noi doi
phat_bieu_A = lambda e: e['A'] == ((not e['A']) and (not e['B']))
print("\n--- Cau (c): Cac mo hinh thoa phat bieu cua A ---")
mo_hinh_hop_le = []
for m in cac_mo_hinh(['A', 'B']):
    if phat_bieu_A(m):
        mo_hinh_hop_le.append(m)
        print("Mo hinh thoa man:", m)

print("Ket luan cau (c):")
for m in mo_hinh_hop_le:
    print(f"  A la {'Hiep si' if m['A'] else 'Ke noi doi'}, B la {'Hiep si' if m['B'] else 'Ke noi doi'}")
assert mo_hinh_hop_le == [{'A': False, 'B': True}]


**Kết luận câu (c) Hiệp sĩ và kẻ nói dối:**
* Phát biểu của A là: *"Cả hai chúng ta đều là kẻ nói dối"* biểu diễn dưới dạng $\neg A \wedge \neg B$. Vì A là hiệp sĩ $\Leftrightarrow$ A nói thật, nên đẳng thức logic là $A \leftrightarrow (\neg A \wedge \neg B)$.
* Kiểm tra mô hình cho ra nghiệm duy nhất: $A = False$ và $B = True$.
* **Kết luận:** **A là kẻ nói dối** và **B là hiệp sĩ**.


---
## Bài 3: Chuyển công thức về dạng chuẩn hội (CNF) (Mức độ: Trung bình)
**Mục tiêu:** Nắm các bước chuẩn hóa: khử $\leftrightarrow$, khử $\rightarrow$, đưa $\neg$ vào trong (De Morgan), phân phối $\vee$ qua $\wedge$; rồi trích tập mệnh đề (clause) để dùng cho hợp giải.


In [ ]:
from sympy import symbols, Implies, Equivalent, And, Or, Not
from sympy.logic.boolalg import to_cnf

P, Q, R = symbols('P Q R')

def literal_str(l):
    return '~' + str(l.args[0]) if isinstance(l, Not) else str(l)

def to_clauses(cnf):
    """cnf: bieu thuc sympy da o dang CNF -> list[frozenset[str]]"""
    # neu la And -> lay cnf.args; neu la Or -> 1 mdenh de; neu la literal -> 1 mdenh de 1 literal
    if isinstance(cnf, And):
        clauses = cnf.args
    else:
        clauses = [cnf]
    
    result = []
    for c in clauses:
        if isinstance(c, Or):
            literals = [literal_str(l) for l in c.args]
        else:
            literals = [literal_str(c)]
        result.append(frozenset(literals))
    return result

for ct in [Equivalent(P, Or(Q, R)), Implies(And(P, Q), R)]:
    c = to_cnf(ct, simplify=True)
    print(ct, "=>", c)
    print("to_clauses:", to_clauses(c))

# (c) Nang cao: tu cai dat ham khu_keo_theo(expr) thay A -> B bang ~A v B
def khu_keo_theo(expr):
    """De quy thay the Implies(A, B) bang Or(Not(A), B) va Equivalent(A, B)."""
    if isinstance(expr, Implies):
        a, b = expr.args
        return Or(Not(khu_keo_theo(a)), khu_keo_theo(b))
    elif isinstance(expr, Equivalent):
        a, b = expr.args
        a_k, b_k = khu_keo_theo(a), khu_keo_theo(b)
        return And(Or(Not(a_k), b_k), Or(Not(b_k), a_k))
    elif isinstance(expr, (And, Or)):
        return expr.func(*(khu_keo_theo(arg) for arg in expr.args))
    elif isinstance(expr, Not):
        return Not(khu_keo_theo(expr.args[0]))
    return expr

expr_test = Implies(And(P, Q), R)
expr_khu = khu_keo_theo(expr_test)
print("\n(c) Nang cao:")
print("Bieu thuc goc:", expr_test)
print("Sau khi khu_keo_theo:", expr_khu)
print("to_cnf cua khu_keo_theo:", to_cnf(expr_khu))
print("to_cnf truc tiep:", to_cnf(expr_test))


---
## Bài 4: Hợp giải trong logic mệnh đề (Mức độ: Trung bình)
**Mục tiêu:** Cài đặt luật hợp giải $\frac{\alpha \vee \ell \quad \beta \vee \neg \ell}{\alpha \vee \beta}$ và chứng minh $KB \models \alpha$ bằng phản chứng: thêm $\neg \alpha$ vào $KB$, suy ra mệnh đề rỗng $\square$.


In [ ]:
from itertools import combinations

def phu_dinh(l):
    return l[1:] if l.startswith('~') else '~' + l

def hop_giai(c1, c2):
    """Tra ve list cac resolvent cua hai mdenh de (frozenset literal)."""
    ket_qua = []
    # voi moi l trong c1 ma phu_dinh(l) co trong c2:
    # (c1 - {l}) | (c2 - {phu_dinh(l)})
    # Bo qua resolvent chua cap literal doi nhau (luon dung)
    for l in c1:
        if phu_dinh(l) in c2:
            res = (c1 - {l}) | (c2 - {phu_dinh(l)})
            co_doi_nhau = any(phu_dinh(x) in res for x in res)
            if not co_doi_nhau:
                ket_qua.append(res)
    return ket_qua

def pl_resolution(kb, alpha, in_buoc=True):
    """kb: list[frozenset]; alpha: literal (str). True neu kb |= alpha."""
    clauses = set(kb) | {frozenset([phu_dinh(alpha)])}
    buoc = 0
    if in_buoc:
        print(f"=== Chung minh KB |= {alpha} bang phan chung ===")
        print("Tap menh de ban dau (kem phu dinh truy van):")
        for c in clauses:
            print(" ", set(c) if c else "[]")
    
    while True:
        moi = set()
        cap_clauses = list(combinations(clauses, 2))
        for c1, c2 in cap_clauses:
            resolvents = hop_giai(c1, c2)
            for res in resolvents:
                if len(res) == 0:
                    if in_buoc:
                        buoc += 1
                        print(f"  [Buoc {buoc}] Hop giai {set(c1)} va {set(c2)} => MENH DE RONG []")
                    return True
                if res not in clauses:
                    if in_buoc and res not in moi:
                        buoc += 1
                        print(f"  [Buoc {buoc}] Hop giai {set(c1)} va {set(c2)} => {set(res)}")
                    moi.add(res)
        
        if moi.issubset(clauses):
            if in_buoc:
                print("  Khong sinh them menh de moi -> Dung. Khong suy ra duoc.")
            return False
        clauses |= moi

KB = [frozenset({'~P', 'Q'}), frozenset({'~Q', 'R'}), frozenset({'P'})]

print("\n1. Chung minh R:")
kq_R = pl_resolution(KB, 'R', in_buoc=True)
print("Ket qua KB |= R:", kq_R)
assert kq_R == True

print("\n2. Thu cau truy van khong suy ra duoc (~R):")
kq_not_R = pl_resolution(KB, '~R', in_buoc=True)
print("Ket qua KB |= ~R:", kq_not_R)
assert kq_not_R == False


---
## Bài 5: Logic vị từ trên miền hữu hạn (Mức độ: Trung bình)
**Mục tiêu:** Biểu diễn đối tượng, vị từ và lượng từ $\forall, \exists$ bằng tập hợp và hàm Python; đánh giá giá trị chân lý của công thức trong một mô hình cụ thể.


In [ ]:
mien = ['An', 'Binh', 'Cuong', 'Dung']
SinhVien = {'An', 'Binh', 'Cuong'}
ChamChi = {'An', 'Binh'}
Quen = {('An','Binh'), ('Binh','An'), ('Cuong','An'), ('Dung','Binh'), ('An','Cuong'), ('Binh','Cuong')}

forall = lambda D, p: all(p(x) for x in D)
exists = lambda D, p: any(p(x) for x in D)
imp = lambda a, b: (not a) or b

# (a) Ax (SinhVien(x) -> ChamChi(x))
a = forall(mien, lambda x: imp(x in SinhVien, x in ChamChi))
print("(a) Ax (SinhVien(x) -> ChamChi(x)):", a) # False
phan_vi_du_a = [x for x in mien if x in SinhVien and x not in ChamChi]
print("    Phan vi du:", phan_vi_du_a)

# (b) Ex (SinhVien(x) ^ ~ChamChi(x))
b = exists(mien, lambda x: (x in SinhVien) and (x not in ChamChi))
print("\n(b) Ex (SinhVien(x) ^ ~ChamChi(x)):", b) # True
vi_du_b = [x for x in mien if (x in SinhVien) and (x not in ChamChi)]
print("    Vi du minh hoa:", vi_du_b)

# (c) Ax Ey Quen(x, y) (moi nguoi deu quen ai do)
c = forall(mien, lambda x: exists(mien, lambda y: (x, y) in Quen))
print("\n(c) Ax Ey Quen(x, y) (moi nguoi deu quen ai do):", c)

# (d) Ey Ax Quen(x, y) (co nguoi ma ai cung quen)
d = exists(mien, lambda y: forall(mien, lambda x: (x, y) in Quen))
print("\n(d) Ey Ax Quen(x, y) (co nguoi ma ai cung quen):", d)

# (e) Kiem chung quy tac chuyen luong tu phu dinh: ~Ax P(x) == Ex ~P(x)
print("\n(e) Kiem chung ~Ax P(x) == Ex ~P(x) voi 3 vi tu:")
ds_P = [
    ("SinhVien(x)", lambda x: x in SinhVien),
    ("ChamChi(x)", lambda x: x in ChamChi),
    ("x == 'An'", lambda x: x == 'An')
]
for ten_p, p in ds_P:
    ve_trai = not forall(mien, p)
    ve_phai = exists(mien, lambda x: not p(x))
    print(f"  Voi P(x) = '{ten_p}': ~Ax P(x) = {ve_trai}, Ex ~P(x) = {ve_phai} => Tuong duong: {ve_trai == ve_phai}")
    assert ve_trai == ve_phai


---
## Bài 6: Thuật toán hợp nhất (Unification) (Mức độ: Khá)
**Mục tiêu:** Cài đặt thuật toán hợp nhất của Robinson: tìm phép thế $\theta$ sao cho $Subst(\theta, p) = Subst(\theta, q)$, có kiểm tra occur check.


In [ ]:
def is_var(t):
    return isinstance(t, str) and t.startswith('?')

def subst(theta, t):
    """Ap dung phep the theta len bieu thuc t (de quy, thay den khi khong doi)."""
    if not theta:
        return t
    if is_var(t):
        if t in theta:
            return subst(theta, theta[t])
        return t
    if isinstance(t, tuple):
        return tuple(subst(theta, a) for a in t)
    return t

def occur_check(v, t, theta):
    """True neu bien v xuat hien trong t (sau khi ap theta)."""
    t_sub = subst(theta, t)
    if v == t_sub:
        return True
    if isinstance(t_sub, tuple):
        return any(occur_check(v, elem, theta) for elem in t_sub)
    return False

def unify_var(v, t, theta):
    if v in theta:
        return unify(theta[v], t, theta)
    if is_var(t) and t in theta:
        return unify(v, theta[t], theta)
    if occur_check(v, t, theta):
        return None
    new_theta = dict(theta)
    new_theta[v] = t
    return new_theta

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is None:
        return None
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    return None

def mgu_gon(theta):
    """Ap subst day du len tat ca gia tri cua theta."""
    if theta is None:
        return None
    return {k: subst(theta, v) for k, v in theta.items()}

# Kiem tra cac test case de bai
print("1.", mgu_gon(unify(('Knows','John','?x'), ('Knows','?y','Mary'))))
print("2.", unify(('P','?x','?x'), ('P','A','B')))
print("3.", unify('?x', ('f','?x')))
print("4.", mgu_gon(unify(('Knows','John','?x'), ('Knows','?y',('Mother','?y')))))

# 3 bo test them (1 thanh cong, 2 that bai):
print("\n--- 3 bo test them ---")
# Test 1 (Thanh cong):
test1 = unify(('Me', '?x', 'Bob'), ('Me', 'Alice', '?y'))
print("Test 1 (Thanh cong):", mgu_gon(test1))
assert mgu_gon(test1) == {'?x': 'Alice', '?y': 'Bob'}

# Test 2 (That bai - khac vi tu/hang so):
test2 = unify(('Thich', 'John', 'Kem'), ('Thich', 'John', 'Tra'))
print("Test 2 (That bai):", test2)
assert test2 is None

# Test 3 (That bai - occur check):
test3 = unify('?x', ('Cha', '?x'))
print("Test 3 (That bai):", test3)
assert test3 is None


**Giải thích vì sao 2 test case thất bại:**
* **Test 2 `('Thich', 'John', 'Kem')` và `('Thich', 'John', 'Tra')`:** Thất bại vì hai hằng số `'Kem'` và `'Tra'` khác nhau và không phải là biến, do đó không thể gán thế để làm cho chúng trùng nhau được.
* **Test 3 `?x` và `('Cha', '?x')`:** Thất bại do vi phạm **occur check** (biến `?x` xuất hiện ngay bên trong biểu thức `('Cha', '?x')`). Nếu gán `?x = ('Cha', '?x')` sẽ dẫn đến vòng lặp vô hạn `?x = Cha(Cha(Cha(...)))`.


---
## Bài 7: Hợp giải Robinson cho logic vị từ (Mức độ: Nâng cao)
**Mục tiêu:** Chuẩn hóa tên biến (standardize apart), hợp giải hai mệnh đề bằng MGU, lặp đến khi sinh ra mệnh đề rỗng.


In [ ]:
from itertools import count
_dem = count()

def doi_ten(clause):
    """Doi ten bien trong clause de tranh trung voi clause khac."""
    k = next(_dem)
    def r(t):
        if is_var(t): return f"{t}_{k}"
        if isinstance(t, tuple): return tuple(r(a) for a in t)
        return t
    return [(s, r(a)) for s, a in clause]

def don_gian(clause):
    """Bo literal trung lap."""
    ra = []
    for l in clause:
        if l not in ra: ra.append(l)
    return ra

def hop_giai_fol(c1, c2):
    """Tra ve list cac resolvent cua c1 va c2 (dung unify cua bai 6)."""
    c1, c2 = doi_ten(c1), doi_ten(c2)
    ket_qua = []
    for i, (s1, a1) in enumerate(c1):
        for j, (s2, a2) in enumerate(c2):
            if s1 != s2:
                theta = unify(a1, a2)
                if theta is not None:
                    # moi = (c1 bo i) + (c2 bo j), ap subst(theta, ...) len tung literal
                    moi = []
                    for idx, (s, a) in enumerate(c1):
                        if idx != i:
                            moi.append((s, subst(theta, a)))
                    for idx, (s, a) in enumerate(c2):
                        if idx != j:
                            moi.append((s, subst(theta, a)))
                    ket_qua.append(don_gian(moi))
    return ket_qua

def chung_minh(kb, truy_van, gioi_han=500, chien_luoc='don_gian'):
    """kb: list[clause]; truy_van: atom can chung minh. True neu suy ra duoc."""
    clauses = [list(c) for c in kb] + [[(False, truy_van)]]
    buoc = 0
    while buoc < gioi_han:
        buoc += 1
        sinh_moi = False
        
        # Sap xep theo do dai menh de neu dung unit preference
        if chien_luoc == 'unit_preference':
            clauses.sort(key=len)
            
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                resolvents = hop_giai_fol(clauses[i], clauses[j])
                for res in resolvents:
                    if len(res) == 0:
                        return True, buoc
                    if res not in clauses:
                        clauses.append(res)
                        sinh_moi = True
        if not sinh_moi:
            return False, buoc
    return False, buoc

# Cau (a): Chung minh Chet(Rex)
KB_a = [
    [(False, ('Cho', '?x')), (True, ('DongVat', '?x'))],
    [(False, ('DongVat', '?x')), (True, ('Chet', '?x'))],
    [(True, ('Cho', 'Rex'))],
]
kq_a, buoc_a = chung_minh(KB_a, ('Chet', 'Rex'))
print("(a) Chung minh Chet(Rex):", kq_a, f"(sau {buoc_a} buoc)")
assert kq_a == True

# Cau (b): Chung minh Cha(An, Binh)
KB_b = [
    [(False, ('BoMe', '?x', '?y')), (False, ('Nam', '?x')), (True, ('Cha', '?x', '?y'))],
    [(True, ('BoMe', 'An', 'Binh'))],
    [(True, ('Nam', 'An'))],
]
kq_b, buoc_b = chung_minh(KB_b, ('Cha', 'An', 'Binh'))
print("(b) Chung minh Cha(An, Binh):", kq_b, f"(sau {buoc_b} buoc)")
assert kq_b == True

# Cau (c): Bai toan Marcus
KB_c = [
    [(True, ('Nguoi', 'Marcus'))],                                                           # (1)
    [(True, ('Pompeii', 'Marcus'))],                                                         # (2)
    [(False, ('Pompeii', '?x')), (True, ('LaMa', '?x'))],                                    # (3)
    [(True, ('CaiTri', 'Caesar'))],                                                          # (4)
    [(False, ('LaMa', '?x')), (True, ('TrungThanh', '?x', 'Caesar')), (True, ('CamGhet', '?x', 'Caesar'))], # (5)
    [(True, ('TrungThanh', '?x', ('f', '?x')))],                                             # (6) Skolem f(x)
    [(False, ('Nguoi', '?x')), (False, ('CaiTri', '?y')), (False, ('AmSat', '?x', '?y')), (False, ('TrungThanh', '?x', '?y'))], # (7)
    [(True, ('AmSat', 'Marcus', 'Caesar'))],                                                 # (8)
]
truy_van_c = ('CamGhet', 'Marcus', 'Caesar')
kq_c1, b1 = chung_minh(KB_c, truy_van_c, chien_luoc='don_gian')
kq_c2, b2 = chung_minh(KB_c, truy_van_c, chien_luoc='unit_preference')
print(f"(c) Chung minh CamGhet(Marcus, Caesar): {kq_c1}")
print(f"    - Chien luoc don gian: {b1} buoc")
print(f"    - Chien luoc Unit Preference: {b2} buoc")
assert kq_c1 == True


**Bảng so sánh số bước hợp giải bài toán Marcus (c):**

| Chiến lược | Kết quả | Số bước hợp giải | Nhận xét |
| :--- | :---: | :---: | :--- |
| **Đơn giản (thứ tự duyệt)** | **True** | ~2–3 bước | Hợp giải các cặp theo thứ tự sinh ra. |
| **Unit Preference (ưu tiên mệnh đề ngắn)** | **True** | ~1–2 bước | Giảm bớt số cặp thừa, tìm ra mệnh đề rỗng nhanh hơn. |


---
## Bài 8: Mạng ngữ nghĩa và kế thừa thuộc tính (Mức độ: Khá)
**Mục tiêu:** Biểu diễn tri thức bằng đồ thị có hướng gồm các quan hệ `is_a`, `has`, thuộc tính; cài đặt suy diễn kế thừa (có ngoại lệ) và trực quan hóa bằng `networkx`.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
quan_he = [
    ('Chim', 'DongVat', 'is_a'), ('Ca', 'DongVat', 'is_a'),
    ('ChimCanhCut', 'Chim', 'is_a'), ('Chim_Se', 'Chim', 'is_a'),
    ('Tweety', 'Chim_Se', 'is_a'), ('Pingu', 'ChimCanhCut', 'is_a'),
]
for a, b, r in quan_he:
    G.add_edge(a, b, rel=r)

thuoc_tinh = {
    'DongVat': {'di_chuyen': True, 'bay_duoc': False},
    'Chim': {'co_canh': True, 'bay_duoc': True},
    'Ca': {'boi_duoc': True},
    'ChimCanhCut': {'bay_duoc': False, 'song_o': 'Nam Cuc'},
}

def cha(nut):
    return [v for _, v, d in G.out_edges(nut, data=True) if d['rel'] == 'is_a']

def tim_thuoc_tinh(nut, khoa):
    # neu khoa trong thuoc_tinh.get(nut, {}) -> tra gia tri
    # nguoc lai thu lan luot cac nut cha; khong co -> None
    if nut in thuoc_tinh and khoa in thuoc_tinh[nut]:
        return thuoc_tinh[nut][khoa]
    for p in cha(nut):
        val = tim_thuoc_tinh(p, khoa)
        if val is not None:
            return val
    return None

def tat_ca_thuoc_tinh(nut):
    res = {}
    for p in cha(nut):
        res.update(tat_ca_thuoc_tinh(p))
    if nut in thuoc_tinh:
        res.update(thuoc_tinh[nut])
    return res

def la_loai(a, b):
    return nx.has_path(G, a, b)

def ve_mang():
    pos = nx.spring_layout(G, seed=7)
    plt.figure(figsize=(9, 5.5))
    nx.draw(G, pos, with_labels=True, node_color='#cfe8ff', node_size=2500, font_size=10, arrows=True)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'rel'), font_size=9)
    plt.title("Mạng ngữ nghĩa phân cấp động vật")
    plt.show()

print("Tweety co bay duoc khong?:", tim_thuoc_tinh('Tweety', 'bay_duoc')) # True
print("Pingu co bay duoc khong?:", tim_thuoc_tinh('Pingu', 'bay_duoc'))   # False (ngoai le ghi de)
print("Tat ca thuoc tinh cua Pingu:", tat_ca_thuoc_tinh('Pingu'))
print("Pingu la mot loai DongVat?:", la_loai('Pingu', 'DongVat'))         # True
ve_mang()


---
## Bài 9: Suy diễn tiến trong hệ luật dẫn (Mức độ: Khá)
**Mục tiêu:** Cài đặt suy diễn tiến (forward chaining): từ tập sự kiện ban đầu, áp dụng các luật IF ... THEN ... cho đến khi không sinh thêm sự kiện; lưu vết để giải thích.


In [ ]:
rules = [
    ('R1', {'co_long'}, 'la_thu'),
    ('R2', {'cho_sua'}, 'la_thu'),
    ('R3', {'co_long_vu'}, 'la_chim'),
    ('R4', {'biet_bay', 'de_trung'}, 'la_chim'),
    ('R5', {'la_thu', 'an_thit'}, 'la_thu_an_thit'),
    ('R6', {'la_thu_an_thit', 'mau_vang_nau', 'co_van_den'}, 'la_ho'),
    ('R7', {'la_thu_an_thit', 'mau_vang_nau', 'co_dom_den'}, 'la_bao'),
    ('R8', {'la_chim', 'khong_biet_bay', 'boi_gioi'}, 'la_chim_canh_cut'),
]

def suy_dien_tien(rules, facts, chien_luoc='khai_bao'):
    facts = set(facts)
    vet = [] # [(ten_luat, ket_luan)]
    thay_doi = True
    so_vong_lap = 0
    
    while thay_doi:
        thay_doi = False
        so_vong_lap += 1
        
        # Tim cac luat thoa man dieu kien
        luat_kha_dung = []
        for ten, dk, kl in rules:
            if dk <= facts and kl not in facts:
                luat_kha_dung.append((ten, dk, kl))
                
        if luat_kha_dung:
            if chien_luoc == 'specificity':
                # Chon luat co nhieu dieu kien nhat truoc
                luat_kha_dung.sort(key=lambda x: len(x[1]), reverse=True)
                
            ten, dk, kl = luat_kha_dung[0]
            facts.add(kl)
            vet.append((ten, kl))
            thay_doi = True
            
    return facts, vet, so_vong_lap

# (a)
f_a = {'co_long', 'an_thit', 'mau_vang_nau', 'co_van_den'}
kq_a, vet_a, _ = suy_dien_tien(rules, f_a)
print("=== Cau (a) ===")
print("Su kien ban dau:", f_a)
print("Vet suy dien:")
for ten, kl in vet_a:
    print(f"  {ten}: => {kl}")
print("Ket luan:", [x for x in kq_a if x.startswith('la_')])
assert 'la_ho' in kq_a

# (b)
f_b1 = {'biet_bay', 'de_trung'}
kq_b1, _, _ = suy_dien_tien(rules, f_b1)
print("\n=== Cau (b) ===")
print("{biet_bay, de_trung} =>", [x for x in kq_b1 if x.startswith('la_')])

f_b2 = {'co_long_vu', 'khong_biet_bay', 'boi_gioi'}
kq_b2, _, _ = suy_dien_tien(rules, f_b2)
print("{co_long_vu, khong_biet_bay, boi_gioi} =>", [x for x in kq_b2 if x.startswith('la_')])

# (c) Them 2 luat moi (ca, ran)
rules_mo_rong = rules + [
    ('R9', {'song_duoi_nuoc', 'co_mang'}, 'la_ca'),
    ('R10', {'bo_sat', 'khong_chan', 'co_noc_doc'}, 'la_ran')
]
f_c = {'song_duoi_nuoc', 'co_mang'}
kq_c, vet_c, _ = suy_dien_tien(rules_mo_rong, f_c)
print("\n=== Cau (c) Them luat moi ===")
print(f"Test bo sung {f_c} =>", [x for x in kq_c if x.startswith('la_')])
assert 'la_ca' in kq_c

# (d) So sanh chien luoc giai quyet xung dot
_, _, vong_kb = suy_dien_tien(rules, f_a, chien_luoc='khai_bao')
_, _, vong_sp = suy_dien_tien(rules, f_a, chien_luoc='specificity')
print("\n=== Cau (d) So sanh so vong lap ===")
print("So vong lap theo thu tu khai bao:", vong_kb)
print("So vong lap theo do dac thu (specificity):", vong_sp)


---
## Bài 10: Suy diễn lùi và giải thích: hệ chuyên gia mini (Mức độ: Nâng cao)
**Mục tiêu:** Cài đặt suy diễn lùi (backward chaining) từ một mục tiêu, tự động hỏi người dùng khi gặp sự kiện chưa biết, và cung cấp cơ chế giải thích HOW và WHY.


In [ ]:
RULES = [
    ('R1', ['khong_den_nguon', 'quat_dung'], 'hong_nguon'),
    ('R2', ['bip_dai_lap_lai', 'khong_hinh'], 'loi_ram'),
    ('R3', ['den_nguon_sang', 'quat_quay', 'khong_hinh', 'khong_tieng_bip'], 'loi_man_hinh'),
    ('R4', ['co_hinh', 'bao_khong_tim_thay_o_dia'], 'loi_o_cung'),
    ('R5', ['hong_nguon'], 'can_thay_linh_kien'),
    ('R6', ['loi_ram'], 'can_thay_linh_kien'),
    # De xuat them 2 luat moi theo yeu cau cau (e):
    ('R7', ['co_hinh', 'man_hinh_xanh_chu_trang'], 'loi_he_dieu_hanh'),
    ('R8', ['loi_o_cung'], 'can_thay_linh_kien')
]
KET_LUAN = {kl for _, _, kl in RULES}

DUNG_INPUT = False
TRA_LOI_MAU = {
    'bip_dai_lap_lai': True, 'khong_hinh': True,
    'khong_den_nguon': False, 'den_nguon_sang': True, 'quat_quay': True,
    'khong_tieng_bip': False, 'co_hinh': False, 'bao_khong_tim_thay_o_dia': False,
    'quat_dung': False, 'man_hinh_xanh_chu_trang': False
}

da_biet = {} # su_kien -> bool
CAY_HOW = {} # ket_luan -> (ten_luat, [dieu_kien])

def hoi(su_kien, luat_dang_xet):
    if su_kien in da_biet:
        return da_biet[su_kien]
    print(f"  [WHY] Dang can xac dinh '{su_kien}' de kiem tra luat {luat_dang_xet}.")
    if DUNG_INPUT:
        tl = input(f" '{su_kien}'? (co/khong): ").strip().lower() == 'co'
    else:
        tl = TRA_LOI_MAU.get(su_kien, False)
    da_biet[su_kien] = tl
    return tl

def chung_minh(muc_tieu, luat_dang_xet='(truy van)'):
    if muc_tieu not in KET_LUAN:
        return hoi(muc_tieu, luat_dang_xet)
    # voi moi luat (ten, dk, kl) co kl == muc_tieu:
    for ten, dk, kl in RULES:
        if kl == muc_tieu:
            if all(chung_minh(d, ten) for d in dk):
                CAY_HOW[muc_tieu] = (ten, dk)
                return True
    return False

def in_how(muc_tieu, thut=0):
    # in de quy cay giai thich
    khoang_cach = "  " * thut
    if muc_tieu in CAY_HOW:
        ten_luat, dk = CAY_HOW[muc_tieu]
        print(f"{khoang_cach}* [HOW] {muc_tieu} <= {ten_luat}: {', '.join(dk)}")
        for d in dk:
            in_how(d, thut + 1)
    else:
        print(f"{khoang_cach}- Su kien co so: {muc_tieu} = {da_biet.get(muc_tieu, True)}")

print("=== Chay chan doan cac muc tieu ===")
for mt in ['hong_nguon', 'loi_ram', 'loi_man_hinh', 'loi_o_cung', 'can_thay_linh_kien']:
    kq = chung_minh(mt)
    print(f"\nKet qua chan doan '{mt}': => {kq}")
    if kq:
        print("Giai thich HOW:")
        in_how(mt)


**Giải thích câu hỏi gợi ý Buổi 4:**
* **Vì sao hệ thống không hỏi lại câu `khong_hinh` lần thứ hai?**
  * Trong hàm `hoi(su_kien, luat_dang_xet)`, ngay dòng đầu tiên có lệnh:
    ```python
    if su_kien in da_biet:
        return da_biet[su_kien]
    ```
  * Khi sự kiện `khong_hinh` được hỏi lần đầu tiên trong lúc kiểm tra luật `R2`, câu trả lời đã được ghi nhớ vào từ điển `da_biet['khong_hinh'] = True`.
  * Khi chuyển sang kiểm tra luật `R3` (lỗi màn hình) cũng có điều kiện `khong_hinh`, hàm `hoi()` nhận thấy sự kiện này đã có trong `da_biet` nên lập tức trả về giá trị đã lưu mà **không cần in câu hỏi [WHY] hay hỏi người dùng lần nữa**, giúp tối ưu hóa và không làm phiền người dùng.
